[Home](../../README.md)

### Data Wrangling

This is a demonstration of data wrangling using [Pandas](https://pandas.pydata.org/) the library for data analysis and manipulation.

This Jupyter Notepad demonstrates different processes you can apply to your data to prepare it for feature engineering and model training. For this demonstration we will wrangle the diabetes data set you previewed in the last Jupyter Notebook.

> [!Note]
> None of these processes are destructive to the source CSV as long as you save the modified data to a new CSV.

#### Load the required dependencies

In [1]:
# Import frameworks
import pandas as pd

####  Store the data as a local variable

The data frame is a Pandas object that structures your tabular data into an appropriate format. It loads the complete data in memory so it is now ready for preprocessing.

In [2]:
data_frame = pd.read_csv("2.1.2.domain_properties.csv")

#### Dealing with null values

Null values during data analysis can cause runtime errors and unexpected results. It is important to identify null values and deal with them appropriately before training a model.

The `isnull().sum()` method call returns the null values in any column.

In [3]:
data_frame.isnull().sum()

price                       0
date_sold                   0
suburb                      0
num_bath                    0
num_bed                     0
num_parking                 0
property_size               0
type                        0
suburb_population           0
suburb_median_income        0
suburb_sqkm                 0
suburb_lat                  0
suburb_lng                  0
suburb_elevation            0
cash_rate                   0
property_inflation_index    0
km_from_cbd                 0
dtype: int64

If you have null data there are many ways to deal with the empty/null values. These are the two most common approaches.
1. Remove any row with a null value with a `dropna()` method call.
2. Replace missing values with another value with a `fillna()` method call. Generally, we use mean value for numerical columns because it may cause minimal changes in your mathematical analysis while maintaining the original size of the data.

Students should reflect why this example removes the null 'SEX' but replacing the mean 'Target'?

#### Remove Duplicates

Duplicate data can have detrimental effects on your machine learning models and outcomes, such as reducing data diversity and representativeness, which can lead to overfitting or biased models.

The `duplicated().sum()` method call returns the count of duplicate rows in the data frame.

In [ ]:
data_frame.duplicated().sum()

np.int64(0)

The `drop_duplicates()` method call can be then stored back onto the data_frame variable removing the duplicates.

In [ ]:
data_frame = data_frame.drop_duplicates()
data_frame.duplicated().sum()

#### Replace data

We can run a lambda function on a column to modify its values. For a simple example, let’s convert the Sex to lowercase. To run a function over a complete column, we can use the apply method which iterates over each row and modifies the values.

In [ ]:
data_frame['type'] = data_frame['type'].apply(lambda x: x.lower())
data_frame['type'].head()

0          house
1          house
2          house
3          house
4    vacant land
Name: type, dtype: str

We can check that there are no data entry errors by the `unique()` method call.

In [ ]:
data_frame['type'].unique()

<StringArray>
[                        'house',                   'vacant land',
                     'townhouse',       'apartment / unit / flat',
                 'semi-detached',              'new house & land',
                        'duplex',                         'villa',
                      'new land',                       'terrace',
                        'studio',                'block of units',
              'development site',          'acreage / semi-rural',
 'new apartments / off the plan',                         'rural']
Length: 16, dtype: str

In [ ]:
data_frame['SEX'] = data_frame['SEX'].apply(lambda gender: 'male' if gender.lower() == 'male' else 'female')
data_frame['SEX'].unique()

#### Deleting columns

In [ ]:
# data_frame = data_frame.drop(columns=["cash_rate"])


#### multiple
# data_frame = data_frame.drop(columns=["cash_rate", "another_column"])

#### Remove outliers

Outliers can skew your analysis on numerical columns, and it is important to remove them. We can use the 25th and 75th quartile on numerical data, to get the inter-quartile range. This allows us to estimate an acceptable range, and we can then filter out any values outside this range. Mathematically, outliers are values occurring outside 1.5 times the interquartile range (IQR) from the first quartile (Q1) or third quartile (Q3).

In [ ]:
#get the inter-quartile range on the blood pressure column
print(data_frame['price'].describe())
Q1 = data_frame['price'].quantile(0.25)
Q3 = data_frame['price'].quantile(0.75)
IQR = Q3 - Q1
print(f'Outliers are a price above {Q3 + IQR * 1.5} or below {Q1 - IQR * 1.5}')


count    1.116000e+04
mean     1.675395e+06
std      1.290371e+06
min      2.250000e+05
25%      1.002000e+06
50%      1.388000e+06
75%      2.020000e+06
max      6.000000e+07
Name: price, dtype: float64
Outliers are a price above 3547000.0 or below -525000.0


In [ ]:
# Filter blood pressure within the acceptable range
data_frame = data_frame[(data_frame['BP'] >= Q1 - 1.5 * IQR) & (data_frame['BP'] <= Q3 + 1.5 * IQR)]
print(data_frame['BP'].describe())

#### Scaling features to a common range

Scaling the features makes it easier for machine learning algorithms to find the optimal solution, as the different scales of the features do not influence them.

In [ ]:
scale_feature = 'BP'

#the minimum value with space for outliers
MIN_BP = 55

#the maximum value with space for outliers
MAX_BP = 140

#scale features
data_frame[scale_feature] = [(X - MIN_BP) / (MAX_BP - MIN_BP) for X in data_frame[scale_feature]]

data_frame.describe()

> [!important]
> You need to save the calculations for each dataset you scale for scaling new values for prediction. Use [2.1.2.data.records.md](2.1.2.data.records.md) to record this information.

#### Save the wrangled data to CSV

In [ ]:
data_frame.to_csv('../2.2.Feature_Engineering/2.2.1.wrangled_data.csv', index=False)